In [18]:
#preprocess_summaries
import pandas as pd
import re
import spacy
from tqdm import tqdm

# Load spaCy English model (we use it for stopword removal, tokenization, lemmatization)
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def preprocess_summary(text):
    """
    Clean and preprocess a movie summary according to specified requirements:
    - Lowercase text
    - Remove numbers and special characters
    - Tokenization
    - Stopword removal
    - Lemmatization
    - Remove short/redundant tokens
    """
    # 1. Lowercasing and basic cleanup
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)          # Remove special chars, numbers
    text = re.sub(r'\s+', ' ', text).strip()       # Remove extra spaces

    # 2. Tokenization + Stopword removal + Lemmatization
    doc = nlp(text)
    tokens = [
        token.lemma_ for token in doc
        if not token.is_stop and token.is_alpha and len(token) > 2
    ]

    return ' '.join(tokens)

def load_and_process_summaries(file_path):
    """
    Load the plot summaries file and apply preprocessing.
    """
    # Load tab-separated data with two columns: MovieID and Summary
    df = pd.read_csv(file_path, sep='\t', names=['MovieID', 'Summary'])

    # Apply preprocessing with progress bar
    tqdm.pandas(desc="🔄 Cleaning Summaries")
    df['Cleaned_Summary'] = df['Summary'].progress_apply(preprocess_summary)

    return df

# Run the script
if __name__ == "__main__":
    file_path = "data/plot_summaries.txt"  # Change path if needed
    cleaned_df = load_and_process_summaries(file_path)

    # Save to CSV
    cleaned_df.to_csv("data/cleaned_summaries.csv", index=False)
    print("✅ Preprocessing complete. Cleaned file saved as 'cleaned_summaries.csv'")


ModuleNotFoundError: No module named 'spacy'

In [7]:
import pandas as pd
import ast
import warnings

def extract_genres(file_path):
    """
    Extract genres from movie.metadata.tsv and convert genre codes to English names.
    Args:
        file_path (str): Path to movie.metadata.tsv.
    Returns:
        pd.DataFrame: DataFrame with Movie ID and list of English genre names.
    """
    # Define genre mapping
    genre_mapping = {
        '/m/01jfsb': 'Thriller',
        '/m/06n90': 'Science Fiction',
        '/m/03npn': 'Horror',
        '/m/03k9fj': 'Adventure',
        '/m/0fdjb': 'Action',
        '/m/02kdv5l': 'Romance',
        '/m/09zvmj': 'War',
        '/m/02n4kr': 'Mystery',
        '/m/03bxz7': 'Biography',
        '/m/07s9rl0': 'Drama',
        '/m/0hj3n01': 'Crime',
        '/m/0lsxr': 'Comedy',
        '/m/0glj9q': 'Family',
        '/m/09blyk': 'Fantasy',
        '/m/02hmvc': 'Short Film',
        '/m/06ppq': 'Silent Film',
        '/m/0219x_': 'Western',
        '/m/01g6gs': 'Animation',
        '/m/0hqxf': 'Fantasy',
        '/m/01hmnh': 'Action',
        '/m/03q4nz': 'War',
        '/m/04t36': 'Musical',
        '/m/0hcr': 'Historical',
        '/m/0hj3myq': 'Romantic Comedy',
        '/m/0hj3myc': 'Romantic Comedy',
        '/m/0c3351': 'Crime',
        '/m/02wtdps': 'Documentary',
        '/m/0vgkd': 'Children',
        '/m/0gw5n2f': 'History',
        '/m/0hj3n0w': 'Romance',
        '/m/01t_vv': 'Sports',
        '/m/068d7h': 'Family',
        '/m/02l7c8': 'Comedy',
        '/m/04xvh5': 'Action',
        '/m/082gq': 'Documentary',
        '/m/06l3bl': 'Adventure',
        '/m/04xvlr': 'Biography',
        '/m/060__y': 'Crime',
        '/m/05p553': 'Romance',
        '/m/0hj3mws': 'Action',
        '/m/0hj3n6f': 'Adventure',
        '/m/0279xh5': 'Science Fiction',
        '/m/0hj3nbk': 'Crime',
        '/m/01z02hx': 'Comedy',
        '/m/02h8pkk': 'Romance',
        '/m/015w9s': 'Horror',
        '/m/0gw5w78': 'Comedy',
        '/m/03btsm8': 'Thriller',
        '/m/01q03': 'Action',
        '/m/0hj3msd': 'War',
        '/m/09q17': 'Animation',
        '/m/0hj3n4b': 'Science Fiction',
        '/m/0hn10': 'Science Fiction',
        '/m/017fp': 'Documentary',
        '/m/0jtdp': 'Documentary',
        '/m/0gf28': 'Science Fiction',
        '/m/0l4h_': 'Children',
        '/m/068twy': 'Short Film',
        '/m/08322': 'Thriller',
        '/m/04t2t': 'Thriller',
        '/m/0gw5qqq': 'War',
        '/m/01z4y': 'Comedy',
        '/m/0hfjk': 'Western',
        '/m/0253g1': 'Short Film',
        '/m/0n6m8vw': 'Romance',
        '/m/03hn0': 'Drama',
        '/m/018td': 'Thriller',
        '/m/01chg': 'Musical',
        '/m/02p0szs': 'Biography',
        '/m/05bh16v': 'Documentary',
        '/m/01j1n2': 'Comedy',
        '/m/0cq22f9': 'Thriller',
        '/m/0cq22z7': 'Horror',
        '/m/0hj3m_q': 'Action',
        '/m/06nbt': 'Action',
        '/m/0jb4p32': 'Horror',
        '/m/01585b': 'Horror',
        '/m/02b5_l': 'Horror',
        '/m/0hj3n7f': 'Action',
        '/m/0424mc': 'Sports',
        '/m/073_6': 'Sports',
        '/m/0q9mp': 'Children',
        '/m/0k345': 'Sports',
        '/m/0bkbm': 'Drama',
        '/m/01yldk': 'Documentary',
        '/m/06b0n3': 'Documentary',
        '/m/0hj3mz0': 'Adventure',
        '/m/01lrrt': 'Comedy',
        '/m/06lbpz': 'Romance',
        '/m/03p5xs': 'Comedy',
        '/m/0hj3mz5': 'Comedy',
        '/m/0qdzd': 'Horror',
        '/m/01jw2w': 'Science Fiction',
        '/m/0bbc17': 'Science Fiction',
        '/m/04q6sch': 'Horror',
        '/m/075qx_b': 'Musical',
        '/m/026ny': 'Crime',
        '/m/0d2rhq': 'Documentary',
        '/m/075fzd': 'Western',
        '/m/0hj3mxx': 'Short Film',
        '/m/01f9r0': 'History',
        '/m/094ddt': 'Thriller',
        '/m/03p15s3': 'Thriller',
        '/m/028v3': 'Comedy',
        '/m/0vjs6': 'Comedy',
        '/m/0gs6m': 'Comedy',
        '/m/03pgfj': 'Crime',
        '/m/0gxblw4': 'Thriller',
        '/m/0j1d47h': 'Romance',
        '/m/04pbhw': 'Thriller',
        '/m/0771_': 'Thriller',
        '/m/0hj3mtj': 'Musical',
        '/m/0cshrf': 'Documentary',
        '/m/05mrx8': 'Action',
        '/m/02qfv5d': 'Crime',
        '/m/0hj3n16': 'History',
        '/m/02js9': 'History',
        '/m/0bj8m2': 'History',
        '/m/0hj3n2k': 'Thriller',
        '/m/04xvlr': 'Biography',
        '/m/0hj3n4p': 'Action',
        '/m/01drsx': 'Thriller',
        '/m/0hj3m_c': 'Action',
        '/m/0hj3nbs': 'Thriller',
        '/m/0556j8': 'Thriller',
        '/m/0hj3l_y': 'Thriller',
        '/m/0dz8b': 'Science Fiction',
        '/m/0rbb': 'Comedy',
        '/m/0hj3n9c': 'Drama',
        '/m/06qln': 'Historical',
        '/m/0bps_n': 'Musical',
        '/m/04btyz': 'Thriller',
        '/m/0hj3n2s': 'Comedy',
        '/m/0d63kt': 'Sports',
        '/m/0bc42t_': 'Sports',
        '/m/04tkhfk': 'Sports',
        '/m/0d1w9': 'Comedy',
        '/m/0hj3mzb': 'Crime',
        '/m/05c4g7': 'Crime',
        '/m/0hj3n07': 'Documentary',
        '/m/04dnp5': 'Documentary',
        '/m/04228s': 'History',
        '/m/0q00t': 'Comedy',
        '/m/05r6t': 'Music',
        '/m/01cgz': 'Comedy',
        '/m/02rd8h3': 'Musical',
        '/m/087lqx': 'Musical',
        '/m/03j0dp': 'Crime',
        '/m/0hj3mwy': 'Science Fiction',
        '/m/04k2v3': 'History',
        '/m/0hj3mt0': 'Science Fiction',
        '/m/0cq23f0': 'Science Fiction',
        '/m/06vxwl5': 'Action',
        '/m/0g092b': 'Action',
        '/m/0fx2s': 'Action',
        '/m/0hj3m_x': 'Comedy',
        '/m/0hj3n1c': 'Science Fiction',
        '/m/0glj9q': 'Family',
        '/m/0gw5n2f': 'History',
        '/m/0jxy': 'Historical',
        '/m/0hj3myc': 'Romantic Comedy',
        '/m/01jk9n': 'History',
        '/m/04gm78f': 'History',
        '/m/0220p9g': 'Musical',
        '/m/0hj3n26': 'Romantic Comedy',
        '/m/02xh1': 'Crime',
        '/m/0fm28': 'Science Fiction',
        '/m/06qm3': 'Comedy',
        '/m/0hj3nbk': 'Crime',
        '/m/0hj3n2s': 'Comedy'
    }

    # Read the metadata file
    try:
        metadata = pd.read_csv(file_path, sep='\t', header=None)
    except FileNotFoundError:
        print(f"Error: File {file_path} not found.")
        return pd.DataFrame(columns=['MovieID', 'Genres_List'])
    except Exception as e:
        print(f"Error reading file: {e}")
        return pd.DataFrame(columns=['MovieID', 'Genres_List'])

    # Assign column names based on dataset structure
    columns = [
        'MovieID', 'FreebaseID', 'Title', 'ReleaseDate', 'BoxOffice',
        'Runtime', 'Language', 'Countries', 'Genres'
    ]
    metadata.columns = columns
    
    # Function to parse genres and convert to English names
    def parse_genres(genre_str):
        try:
            # Safely evaluate the string as a Python dictionary
            genre_dict = ast.literal_eval(genre_str)
            if not isinstance(genre_dict, dict):
                raise ValueError("Genres is not a dictionary")
            
            # Extract genre codes and convert to English names
            english_genres = []
            for code in genre_dict.keys():
                if code in genre_mapping:
                    english_genres.append(genre_mapping[code])
                else:
                    warnings.warn(f"Genre code {code} not found in mapping. Marking as Unknown.")
                    english_genres.append(f"Unknown_{code}")
            
            return english_genres
        except (ValueError, SyntaxError) as e:
            warnings.warn(f"Error parsing Genres: {genre_str}. Error: {e}")
            return []
    
    # Extract and convert genres
    metadata['Genres_List'] = metadata['Genres'].apply(parse_genres)
    
    # Select relevant columns
    genres_df = metadata[['MovieID', 'Genres_List']]
    
    return genres_df

# Example usage
if __name__ == "__main__":
    metadata_file = "data/movie.metadata.tsv"
    genres_df = extract_genres(metadata_file)
    
    # Save genres to CSV
    try:
        genres_df.to_csv("data/genres.csv", index=False)
        print("Genres saved to 'data/genres.csv'.")
    except Exception as e:
        print(f"Error saving CSV: {e}")

Genres saved to 'data/genres.csv'.


In [8]:
#create_cleaned_dataset
import pandas as pd

def create_cleaned_dataset(summaries_file, genres_file, output_file):
    """
    Combine cleaned summaries and genres into a single dataset.
    Args:
        summaries_file (str): Path to cleaned_summaries.csv.
        genres_file (str): Path to genres.csv.
        output_file (str): Path to save the final dataset.
    Returns:
        pd.DataFrame: Combined dataset with Movie ID, cleaned summary, and genres.
    """
    # Load cleaned summaries
    summaries_df = pd.read_csv(summaries_file)
    
    # Load genres
    genres_df = pd.read_csv(genres_file)
    
    # Merge on MovieID
    combined_df = pd.merge(
        summaries_df[['MovieID', 'Cleaned_Summary']],
        genres_df[['MovieID', 'Genres_List']],
        on='MovieID',
        how='inner'
    )
    
    # Drop rows with empty genres or summaries
    combined_df = combined_df[combined_df['Cleaned_Summary'].notna() & (combined_df['Genres_List'].str.len() > 0)]
    
    # Save to CSV
    combined_df.to_csv(output_file, index=False)
    
    return combined_df

# Example usage
if __name__ == "__main__":
    summaries_file = "data/cleaned_summaries.csv"
    genres_file = "data/genres.csv"
    output_file = "data/cleaned_dataset.csv"
    
    cleaned_dataset = create_cleaned_dataset(summaries_file, genres_file, output_file)
    print(f"Cleaned dataset saved to '{output_file}' with {len(cleaned_dataset)} records.")

Cleaned dataset saved to 'data/cleaned_dataset.csv' with 42204 records.


In [9]:
#train_test_split
import pandas as pd
from sklearn.model_selection import train_test_split

def perform_train_test_split(input_file, train_file, test_file, test_size=0.2, random_state=42):
    """
    Split the cleaned dataset into training and testing sets.
    Args:
        input_file (str): Path to cleaned_dataset.csv.
        train_file (str): Path to save training set.
        test_file (str): Path to save testing set.
        test_size (float): Proportion of dataset to use for testing.
        random_state (int): Seed for reproducibility.
    Returns:
        tuple: (train_df, test_df) DataFrames for training and testing.
    """
    # Load cleaned dataset
    df = pd.read_csv(input_file)
    
    # Perform train-test split
    train_df, test_df = train_test_split(
        df,
        test_size=test_size,
        random_state=random_state
    )
    
    # Save training and testing sets
    train_df.to_csv(train_file, index=False)
    test_df.to_csv(test_file, index=False)
    
    return train_df, test_df

# Example usage
if __name__ == "__main__":
    input_file = "data/cleaned_dataset.csv"
    train_file = "data/train_dataset.csv"
    test_file = "data/test_dataset.csv"
    
    train_df, test_df = perform_train_test_split(input_file, train_file, test_file)
    print(f"Training set saved to '{train_file}' with {len(train_df)} records.")
    print(f"Testing set saved to '{test_file}' with {len(test_df)} records.")

Training set saved to 'data/train_dataset.csv' with 33763 records.
Testing set saved to 'data/test_dataset.csv' with 8441 records.


In [15]:
import pandas as pd
from deep_translator import GoogleTranslator
from tqdm import tqdm
import time
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def split_text(text, max_len=5000):
    """
    Splits text into chunks of max_len characters without cutting words.
    Args:
        text (str): Text to split.
        max_len (int): Maximum length per chunk.
    Returns:
        list: List of text chunks.
    """
    if not isinstance(text, str):
        logger.warning("Non-string input to split_text. Returning empty list.")
        return []
    
    chunks = []
    while len(text) > max_len:
        split_index = text.rfind(' ', 0, max_len)
        if split_index == -1:
            # No space found; split at max_len
            split_index = max_len
        chunks.append(text[:split_index].strip())
        text = text[split_index:].strip()
    if text:
        chunks.append(text)
    return chunks

def translate_text(text, lang_code, max_retries=3):
    """
    Safely translates text, handling chunking and API errors with retries.
    Args:
        text (str): Text to translate.
        lang_code (str): Target language code (e.g., 'ar').
        max_retries (int): Number of retry attempts.
    Returns:
        str: Translated text or empty string on failure.
    """
    translator = GoogleTranslator(source='auto', target=lang_code)
    chunks = split_text(text)
    translated_chunks = []
    
    for chunk in chunks:
        for attempt in range(max_retries):
            try:
                translated_chunk = translator.translate(chunk)
                if translated_chunk is None:
                    raise ValueError("Translation returned None")
                translated_chunks.append(translated_chunk)
                time.sleep(1.5)  # Increased delay to avoid rate-limiting
                break
            except Exception as e:
                logger.error(f"Attempt {attempt + 1} failed for chunk '{chunk[:50]}...': {e}")
                if attempt == max_retries - 1:
                    logger.error(f"Max retries reached for chunk. Skipping.")
                    return ''
                time.sleep(2 ** attempt)  # Exponential backoff
    return ' '.join(translated_chunks)

def translate_summaries(input_file, output_file, num_summaries=50):
    """
    Translate movie summaries into Arabic, Urdu, and Korean using deep_translator.
    Args:
        input_file (str): Path to input CSV with cleaned summaries.
        output_file (str): Path to output CSV for translated summaries.
        num_summaries (int): Number of summaries to process.
    Returns:
        pd.DataFrame: DataFrame with translated summaries.
    """
    # Read input file
    try:
        df = pd.read_csv(input_file)
    except FileNotFoundError:
        logger.error(f"Input file {input_file} not found.")
        return pd.DataFrame()
    except Exception as e:
        logger.error(f"Error reading input file: {e}")
        return pd.DataFrame()

    # Validate input
    if 'Cleaned_Summary' not in df.columns:
        logger.error("Input CSV must contain 'Cleaned_Summary' column.")
        return pd.DataFrame()
    
    df = df.head(num_summaries).copy()
    if df.empty:
        logger.warning("No summaries to process.")
        return df

    # Initialize translation columns
    languages = {'Arabic': 'ar', 'Urdu': 'ur', 'Korean': 'ko'}
    for lang in languages:
        df[f'Summary_{lang}'] = ''

    # Translate summaries
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Translating Summaries"):
        summary = str(row['Cleaned_Summary']).strip()
        if not summary or pd.isna(summary) or summary.lower() == 'nan':
            logger.info(f"Skipping empty or invalid summary at index {idx}")
            continue
        
        for lang_name, lang_code in languages.items():
            translated = translate_text(summary, lang_code)
            df.at[idx, f'Summary_{lang_name}'] = translated if translated else 'Translation Failed'
            
            # Save partial results to avoid data loss
            try:
                df.to_csv(output_file, index=False)
            except Exception as e:
                logger.error(f"Error saving partial results: {e}")

    # Final save
    try:
        df.to_csv(output_file, index=False)
        logger.info(f"Translated summaries saved to '{output_file}' with {len(df)} records.")
    except Exception as e:
        logger.error(f"Error saving final CSV: {e}")

    return df

if __name__ == "__main__":
    input_file = "data/cleaned_dataset.csv"
    output_file = "data/translated_summaries.csv"
    translated_df = translate_summaries(input_file, output_file)
    print(f"\n✅ Translated summaries saved to '{output_file}' with {len(translated_df)} records.")

Translating Summaries: 100%|██████████| 50/50 [07:06<00:00,  8.54s/it]
2025-05-06 13:50:54,490 - INFO - Translated summaries saved to 'data/translated_summaries.csv' with 50 records.



✅ Translated summaries saved to 'data/translated_summaries.csv' with 50 records.


In [16]:
#convert to audio
import pandas as pd
from gtts import gTTS
import os
import time

def convert_to_audio(input_file, audio_dir):
    """
    Convert translated summaries to audio files.
    Args:
        input_file (str): Path to translated_summaries.csv.
        audio_dir (str): Directory to save audio files.
    """
    # Load translated summaries
    df = pd.read_csv(input_file)
    
    # Create output directory
    os.makedirs(audio_dir, exist_ok=True)
    
    # Define languages
    languages = {'Arabic': 'ar', 'Urdu': 'ur', 'Korean': 'ko'}
    
    # Convert each summary to audio
    for idx, row in df.iterrows():
        movie_id = row['MovieID']
        # Create subdirectory for each MovieID
        movie_dir = os.path.join(audio_dir, str(movie_id))
        os.makedirs(movie_dir, exist_ok=True)
        
        for lang_name, lang_code in languages.items():
            text = row[f'Summary_{lang_name}']
            if not text or pd.isna(text):
                print(f"Skipping empty translation for MovieID {movie_id}, {lang_name}")
                continue
            try:
                tts = gTTS(text=text, lang=lang_code)
                output_file = os.path.join(movie_dir, f"{lang_name.lower()}.mp3")
                tts.save(output_file)
                print(f"Generated audio: {output_file}")
                time.sleep(1)  # Avoid API rate limits
            except Exception as e:
                print(f"Error generating audio for MovieID {movie_id}, {lang_name}: {e}")

if __name__ == "__main__":
    input_file = "data/translated_summaries.csv"
    audio_dir = "audio/summaries"
    
    convert_to_audio(input_file, audio_dir)
    print(f"Audio files saved in '{audio_dir}'.")

Error generating audio for MovieID 23890098, Arabic: 429 (Too Many Requests) from TTS API. Probable cause: Unknown
Error generating audio for MovieID 23890098, Urdu: 429 (Too Many Requests) from TTS API. Probable cause: Unknown
Error generating audio for MovieID 23890098, Korean: 429 (Too Many Requests) from TTS API. Probable cause: Unknown
Error generating audio for MovieID 31186339, Arabic: 429 (Too Many Requests) from TTS API. Probable cause: Unknown
Error generating audio for MovieID 31186339, Urdu: 429 (Too Many Requests) from TTS API. Probable cause: Unknown
Error generating audio for MovieID 31186339, Korean: 429 (Too Many Requests) from TTS API. Probable cause: Unknown
Error generating audio for MovieID 20663735, Arabic: 429 (Too Many Requests) from TTS API. Probable cause: Unknown
Error generating audio for MovieID 20663735, Urdu: 429 (Too Many Requests) from TTS API. Probable cause: Unknown
Error generating audio for MovieID 20663735, Korean: 429 (Too Many Requests) from TTS 

KeyboardInterrupt: 

In [ ]:
#audio playback
import pandas as pd
import os
from playsound import playsound

def audio_playback(translated_file, audio_dir):
    """
    Menu-based system for playing audio files.
    Args:
        translated_file (str): Path to translated_summaries.csv.
        audio_dir (str): Directory containing audio files.
    """
    # Load translated summaries
    df = pd.read_csv(translated_file)
    
    # List available MovieIDs
    movie_ids = df['MovieID'].tolist()
    
    while True:
        print("\nAvailable MovieIDs:", movie_ids)
        print("Enter 'q' to quit.")
        movie_id = input("Enter MovieID: ")
        
        if movie_id.lower() == 'q':
            break
        
        if int(movie_id) not in movie_ids:
            print("Invalid MovieID. Try again.")
            continue
        
        # Display language options
        languages = ['Arabic', 'Urdu', 'Korean']
        print("\nAvailable languages:", languages)
        lang = input("Enter language: ").capitalize()
        
        if lang not in languages:
            print("Invalid language. Try again.")
            continue
        
        # Construct audio file path
        audio_file = os.path.join(audio_dir, str(movie_id), f"{lang.lower()}.mp3")
        
        if not os.path.exists(audio_file):
            print(f"Audio file not found for MovieID {movie_id}, {lang}.")
            continue
        
        try:
            print(f"Playing audio: {audio_file}")
            playsound(audio_file)
        except Exception as e:
            print(f"Error playing audio: {e}")

if __name__ == "__main__":
    translated_file = "data/translated_summaries.csv"
    audio_dir = "audio/summaries"
    
    audio_playback(translated_file, audio_dir)

In [ ]:
#task3
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle
import os

def extract_tfidf_features(train_file, test_file, vectorizer_file):
    """
    Extract TF-IDF features from movie summaries.
    Args:
        train_file (str): Path to train_dataset.csv.
        test_file (str): Path to test_dataset.csv.
        vectorizer_file (str): Path to save the fitted TfidfVectorizer.
    Returns:
        tuple: (X_train, X_test, vectorizer) TF-IDF features and vectorizer.
    """
    # Load datasets
    train_df = pd.read_csv(train_file)
    test_df = pd.read_csv(test_file)
    
    # Extract summaries
    train_summaries = train_df['Cleaned_Summary'].fillna('')
    test_summaries = test_df['Cleaned_Summary'].fillna('')
    
    # Initialize TF-IDF vectorizer
    vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
    
    # Fit and transform on training data
    X_train = vectorizer.fit_transform(train_summaries)
    
    # Transform test data (using the same vectorizer)
    X_test = vectorizer.transform(test_summaries)
    
    # Save the vectorizer for later use
    with open(vectorizer_file, 'wb') as f:
        pickle.dump(vectorizer, f)
    
    return X_train, X_test, vectorizer

if __name__ == "__main__":
    train_file = "data/train_dataset.csv"
    test_file = "data/test_dataset.csv"
    vectorizer_file = "models/tfidf_vectorizer.pkl"
    
    os.makedirs("models", exist_ok=True)
    X_train, X_test, vectorizer = extract_tfidf_features(train_file, test_file, vectorizer_file)
    print(f"TF-IDF features extracted: X_train shape = {X_train.shape}, X_test shape = {X_test.shape}")
    print(f"Vectorizer saved to '{vectorizer_file}'.")

In [17]:
#train model
import os
import ast
import pickle
import pandas as pd
from extract_features import extract_tfidf_features
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

def prepare_labels(train_file, test_file, mlb_file):
    """
    Prepare multi-label genre labels.
    Args:
        train_file (str): Path to train_dataset.csv.
        test_file (str): Path to test_dataset.csv.
        mlb_file (str): Path to save the fitted MultiLabelBinarizer.
    Returns:
        tuple: (y_train, y_test, mlb) Binary labels and binarizer.
    """
    # Load datasets
    train_df = pd.read_csv(train_file)
    test_df = pd.read_csv(test_file)
    
    # Convert string representations of lists to actual lists
    train_df['Genres_List'] = train_df['Genres_List'].apply(ast.literal_eval)
    test_df['Genres_List'] = test_df['Genres_List'].apply(ast.literal_eval)
    
    # Initialize MultiLabelBinarizer
    mlb = MultiLabelBinarizer()
    
    # Fit and transform on training genres
    y_train = mlb.fit_transform(train_df['Genres_List'])
    
    # Transform test genres
    y_test = mlb.transform(test_df['Genres_List'])
    
    # Save the binarizer
    with open(mlb_file, 'wb') as f:
        pickle.dump(mlb, f)
    
    return y_train, y_test, mlb

def train_model(X_train, y_train, model_file):
    """
    Train a multi-label Logistic Regression model.
    Args:
        X_train: TF-IDF features for training.
        y_train: Binary genre labels for training.
        model_file (str): Path to save the trained model.
    Returns:
        OneVsRestClassifier: Trained model.
    """
    # Initialize Logistic Regression with OneVsRestClassifier
    model = OneVsRestClassifier(LogisticRegression(max_iter=1000))
    
    # Train the model
    model.fit(X_train, y_train)
    
    # Save the model
    with open(model_file, 'wb') as f:
        pickle.dump(model, f)
    
    return model

if __name__ == "__main__":
    train_file = "data/train_dataset.csv"
    test_file = "data/test_dataset.csv"
    mlb_file = "models/mlb.pkl"
    model_file = "models/logistic_regression_model.pkl"
    
    os.makedirs("models", exist_ok=True)
    
    # Load TF-IDF features (assumes extract_features.py has been run)
    X_train, X_test, vectorizer = extract_tfidf_features(train_file, test_file, "models/tfidf_vectorizer.pkl")
    
    # Prepare labels
    y_train, y_test, mlb = prepare_labels(train_file, test_file, mlb_file)
    print(f"Labels prepared: y_train shape = {y_train.shape}, y_test shape = {y_test.shape}")
    
    # Train model
    model = train_model(X_train, y_train, model_file)
    print(f"Model trained and saved to '{model_file}'.")

Labels prepared: y_train shape = (33763, 221), y_test shape = (8441, 221)
Model trained and saved to 'models/logistic_regression_model.pkl'.


In [ ]:
#evaluate model
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import MultiLabelBinarizer
import joblib
import warnings

# Ignore sklearn warnings about unknown labels during transformation
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

def load_data():
    X_train = joblib.load("X_train.pkl")
    y_train = joblib.load("y_train.pkl")
    X_test = joblib.load("X_test.pkl")
    y_test = joblib.load("y_test.pkl")
    return X_train, y_train, X_test, y_test

def load_model_and_mlb():
    model = joblib.load("model.pkl")
    mlb = joblib.load("mlb.pkl")
    return model, mlb

def print_metrics(name, y_true, y_pred):
    print(f"{name} Metrics:")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision (micro):", precision_score(y_true, y_pred, average="micro", zero_division=0))
    print("Recall (micro):", recall_score(y_true, y_pred, average="micro", zero_division=0))
    print("F1-Score (micro):", f1_score(y_true, y_pred, average="micro", zero_division=0))
    print()

def evaluate_model(model, X_train, y_train, X_test, y_test, mlb, output_dir="output"):
    os.makedirs(output_dir, exist_ok=True)

    y_train_bin = mlb.transform(y_train)
    y_test_bin = mlb.transform(y_test)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    print_metrics("Training", y_train_bin, y_train_pred)
    print_metrics("Test", y_test_bin, y_test_pred)

    # Compute confusion matrix for each genre
    for i, genre in enumerate(mlb.classes_):
        try:
            cm_train = confusion_matrix(y_train_bin[:, i], y_train_pred[:, i], labels=[0, 1])
            cm_test = confusion_matrix(y_test_bin[:, i], y_test_pred[:, i], labels=[0, 1])

            disp_train = ConfusionMatrixDisplay(confusion_matrix=cm_train, display_labels=[0, 1])
            disp_test = ConfusionMatrixDisplay(confusion_matrix=cm_test, display_labels=[0, 1])

            safe_genre = genre.replace("/", "_").replace("\\", "_").replace(" ", "_")  # sanitize filename
            plt.figure()
            disp_train.plot()
            plt.title(f"Train Confusion Matrix for {genre}")
            plt.savefig(os.path.join(output_dir, f"cm_train_{safe_genre}.png"))
            plt.close()

            plt.figure()
            disp_test.plot()
            plt.title(f"Test Confusion Matrix for {genre}")
            plt.savefig(os.path.join(output_dir, f"cm_test_{safe_genre}.png"))
            plt.close()

        except Exception as e:
            print(f"[Warning] Skipped confusion matrix for '{genre}': {e}")

if __name__ == "__main__":
    X_train, y_train, X_test, y_test = load_data()
    model, mlb = load_model_and_mlb()
    evaluate_model(model, X_train, y_train, X_test, y_test, mlb)


In [ ]:
#menu_system
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from googletrans import Translator
from gtts import gTTS
import pygame
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
import os
import time

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

# Initialize lemmatizer and stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_summary(text):
    """
    Clean and preprocess a movie summary using NLTK.
    Args:
        text (str): Raw movie summary.
    Returns:
        str: Cleaned and preprocessed summary.
    """
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    cleaned_tokens = [
        lemmatizer.lemmatize(token) for token in tokens
        if token not in stop_words and len(token) > 2
    ]
    return ' '.join(cleaned_tokens)

def translate_summary(summary, lang_code):
    """
    Translate a summary into the specified language.
    Args:
        summary (str): Input summary.
        lang_code (str): Target language code (e.g., 'ar', 'ur', 'ko').
    Returns:
        str: Translated summary.
    """
    translator = Translator()
    try:
        translated = translator.translate(summary, dest=lang_code)
        return translated.text
    except Exception as e:
        print(f"Error translating to {lang_code}: {e}")
        return ""

def generate_and_play_audio(summary, lang_code, lang_name):
    """
    Generate and play audio for a summary using pygame.
    Args:
        summary (str): Text to convert to audio.
        lang_code (str): Language code.
        lang_name (str): Language name for file naming.
    """
    if not summary:
        print("No text to convert to audio.")
        return
    
    os.makedirs("temp", exist_ok=True)
    audio_file = f"temp/{lang_name.lower()}_summary.mp3"
    
    try:
        tts = gTTS(text=summary, lang=lang_code)
        tts.save(audio_file)
        pygame.mixer.init()
        pygame.mixer.music.load(audio_file)
        print(f"Playing audio: {audio_file}")
        pygame.mixer.music.play()
        while pygame.mixer.music.get_busy():
            time.sleep(0.1)
        pygame.mixer.quit()
        time.sleep(1)  # Avoid API rate limits
    except Exception as e:
        print(f"Error generating or playing audio: {e}")

def predict_genres(summary, vectorizer, model, mlb):
    """
    Predict genres for a given summary.
    Args:
        summary (str): Cleaned movie summary.
        vectorizer: Fitted TfidfVectorizer.
        model: Trained OneVsRestClassifier.
        mlb: Fitted MultiLabelBinarizer.
    Returns:
        list: Predicted genres.
    """
    if not summary:
        return []
    
    X = vectorizer.transform([summary])
    y_pred = model.predict(X)
    genres = mlb.inverse_transform(y_pred)[0]
    return list(genres) if genres else ["No genres predicted"]

def main():
    """
    Main function for the menu-based system.
    """
    try:
        with open("models/tfidf_vectorizer.pkl", 'rb') as f:
            vectorizer = pickle.load(f)
        with open("models/mlb.pkl", 'rb') as f:
            mlb = pickle.load(f)
        with open("models/logistic_regression_model.pkl", 'rb') as f:
            model = pickle.load(f)
    except FileNotFoundError as e:
        print(f"Error: Model or vectorizer file not found: {e}")
        return
    
    print("Welcome to Filmception: AI-Powered Movie Summary Translator and Genre Classifier")
    
    # Get user input for summary
    while True:
        print("\nPrompting for summary input...")
        summary = input("Enter a movie summary (or 'q' to quit): ").strip()
        print(f"Received input: '{summary}'")
        if summary.lower() == 'q':
            print("Exiting Filmception. Goodbye!")
            return
        if not summary:
            print("Error: Summary cannot be empty. Please try again.")
            continue
        break
    
    cleaned_summary = clean_summary(summary)
    
    # Menu loop
    languages = {'Arabic': 'ar', 'Urdu': 'ur', 'Korean': 'ko'}
    while True:
        print("\nMenu:")
        print("1. Convert Summary to Audio")
        print("2. Predict Genre")
        print("3. Exit")
        choice = input("Enter your choice (1-3): ").strip()
        
        if choice == '1':
            print("\nAvailable languages:", list(languages.keys()))
            lang_name = input("Enter language: ").capitalize()
            if lang_name not in languages:
                print("Error: Invalid language. Please choose from", list(languages.keys()))
                continue
            
            lang_code = languages[lang_name]
            translated_summary = translate_summary(summary, lang_code)
            if translated_summary:
                print(f"Translated summary ({lang_name}): {translated_summary}")
                generate_and_play_audio(translated_summary, lang_code, lang_name)
            else:
                print("Translation failed. Please try again.")
        
        elif choice == '2':
            genres = predict_genres(cleaned_summary, vectorizer, model, mlb)
            print("\nPredicted Genres:", ", ".join(genres) if genres else "None")
        
        elif choice == '3':
            print("Exiting Filmception. Goodbye!")
            break
        
        else:
            print("Error: Invalid choice. Please enter 1, 2, or 3.")

if __name__ == "__main__":
    main()

In [ ]:
#gui
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from googletrans import Translator
from gtts import gTTS
import pygame
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
import os
import time
import tkinter as tk
from tkinter import messagebox, scrolledtext

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

# Initialize lemmatizer and stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_summary(text):
    """
    Clean and preprocess a movie summary using NLTK.
    Args:
        text (str): Raw movie summary.
    Returns:
        str: Cleaned and preprocessed summary.
    """
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    cleaned_tokens = [
        lemmatizer.lemmatize(token) for token in tokens
        if token not in stop_words and len(token) > 2
    ]
    return ' '.join(cleaned_tokens)

def translate_summary(summary, lang_code):
    """
    Translate a summary into the specified language.
    Args:
        summary (str): Input summary.
        lang_code (str): Target language code (e.g., 'ar', 'ur', 'ko').
    Returns:
        str: Translated summary.
    """
    translator = Translator()
    try:
        translated = translator.translate(summary, dest=lang_code)
        return translated.text
    except Exception as e:
        return f"Error translating to {lang_code}: {e}"

def generate_and_play_audio(summary, lang_code, lang_name):
    """
    Generate and play audio for a summary using pygame.
    Args:
        summary (str): Text to convert to audio.
        lang_code (str): Language code.
        lang_name (str): Language name for file naming.
    Returns:
        str: Success or error message.
    """
    if not summary:
        return "No text to convert to audio."
    
    os.makedirs("temp", exist_ok=True)
    audio_file = f"temp/{lang_name.lower()}_summary.mp3"
    
    try:
        tts = gTTS(text=summary, lang=lang_code)
        tts.save(audio_file)
        pygame.mixer.init()
        pygame.mixer.music.load(audio_file)
        pygame.mixer.music.play()
        while pygame.mixer.music.get_busy():
            time.sleep(0.1)
        pygame.mixer.quit()
        time.sleep(1)  # Avoid API rate limits
        return f"Playing audio: {audio_file}"
    except Exception as e:
        return f"Error generating or playing audio: {e}"

def predict_genres(summary, vectorizer, model, mlb):
    """
    Predict genres for a given summary.
    Args:
        summary (str): Cleaned movie summary.
        vectorizer: Fitted TfidfVectorizer.
        model: Trained OneVsRestClassifier.
        mlb: Fitted MultiLabelBinarizer.
    Returns:
        list: Predicted genres.
    """
    if not summary:
        return ["No genres predicted"]
    
    X = vectorizer.transform([summary])
    y_pred = model.predict(X)
    genres = mlb.inverse_transform(y_pred)[0]
    return list(genres) if genres else ["No genres predicted"]

class FilmceptionGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("Filmception: Movie Summary Translator and Genre Classifier")
        self.root.geometry("600x500")
        
        # Load models
        try:
            with open("models/tfidf_vectorizer.pkl", 'rb') as f:
                self.vectorizer = pickle.load(f)
            with open("models/mlb.pkl", 'rb') as f:
                self.mlb = pickle.load(f)
            with open("models/logistic_regression_model.pkl", 'rb') as f:
                self.model = pickle.load(f)
        except FileNotFoundError as e:
            messagebox.showerror("Error", f"Model or vectorizer file not found: {e}")
            self.root.destroy()
            return
        
        # Language options
        self.languages = {'Arabic': 'ar', 'Urdu': 'ur', 'Korean': 'ko'}
        
        # GUI elements
        tk.Label(root, text="Enter Movie Summary:", font=("Arial", 12)).pack(pady=10)
        
        self.summary_text = scrolledtext.ScrolledText(root, height=5, width=50, wrap=tk.WORD)
        self.summary_text.pack(pady=10)
        
        tk.Label(root, text="Select Action:", font=("Arial", 12)).pack(pady=10)
        
        tk.Button(root, text="Translate and Play Audio (Arabic)", command=lambda: self.process_audio('Arabic')).pack(pady=5)
        tk.Button(root, text="Translate and Play Audio (Urdu)", command=lambda: self.process_audio('Urdu')).pack(pady=5)
        tk.Button(root, text="Translate and Play Audio (Korean)", command=lambda: self.process_audio('Korean')).pack(pady=5)
        tk.Button(root, text="Predict Genres", command=self.predict).pack(pady=5)
        tk.Button(root, text="Exit", command=self.exit).pack(pady=5)
        
        tk.Label(root, text="Output:", font=("Arial", 12)).pack(pady=10)
        self.output_text = scrolledtext.ScrolledText(root, height=8, width=50, wrap=tk.WORD, state='disabled')
        self.output_text.pack(pady=10)
    
    def process_audio(self, lang_name):
        """
        Process summary for translation and audio playback.
        """
        summary = self.summary_text.get("1.0", tk.END).strip()
        if not summary:
            messagebox.showerror("Error", "Summary cannot be empty.")
            return
        
        self.output_text.configure(state='normal')
        self.output_text.delete("1.0", tk.END)
        
        if lang_name not in self.languages:
            self.output_text.insert(tk.END, f"Invalid language: {lang_name}\n")
            self.output_text.configure(state='disabled')
            return
        
        lang_code = self.languages[lang_name]
        translated_summary = translate_summary(summary, lang_code)
        if translated_summary.startswith("Error"):
            self.output_text.insert(tk.END, translated_summary + "\n")
        else:
            self.output_text.insert(tk.END, f"Translated summary ({lang_name}): {translated_summary}\n")
            audio_result = generate_and_play_audio(translated_summary, lang_code, lang_name)
            self.output_text.insert(tk.END, audio_result + "\n")
        
        self.output_text.configure(state='disabled')
    
    def predict(self):
        """
        Predict genres for the input summary.
        """
        summary = self.summary_text.get("1.0", tk.END).strip()
        if not summary:
            messagebox.showerror("Error", "Summary cannot be empty.")
            return
        
        cleaned_summary = clean_summary(summary)
        genres = predict_genres(cleaned_summary, self.vectorizer, self.model, self.mlb)
        
        self.output_text.configure(state='normal')
        self.output_text.delete("1.0", tk.END)
        self.output_text.insert(tk.END, "Predicted Genres: " + ", ".join(genres) + "\n")
        self.output_text.configure(state='disabled')
    
    def exit(self):
        """
        Exit the application.
        """
        self.root.destroy()

if __name__ == "__main__":
    root = tk.Tk()
    app = FilmceptionGUI(root)
    root.mainloop()

In [ ]:
import pandas as pd
import os
import pickle
from googletrans import Translator
from gtts import gTTS
import pygame
import time
import ast
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import re
from collections import Counter

warnings.filterwarnings("ignore")

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

# Initialize lemmatizer and stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Get base directory
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

def clean_summary(text):
    """
    Clean a summary to match Task 1 preprocessing.
    Args:
        text (str): Raw text.
    Returns:
        str: Cleaned text.
    """
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    cleaned_tokens = [
        lemmatizer.lemmatize(token) for token in tokens
        if token not in stop_words and len(token) > 2
    ]
    return ' '.join(cleaned_tokens)

def check_file_exists(file_path, description):
    """
    Check if a file exists and raise a detailed error if not.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"{description} not found at: {file_path}")

def get_top_genres(train_file, top_n=20):
    """
    Identify the top N genres by frequency in the training dataset.
    Args:
        train_file (str): Path to train_dataset.csv.
        top_n (int): Number of top genres to select.
    Returns:
        list: List of top N genre names.
    """
    try:
        train_df = pd.read_csv(train_file)
        train_df['Genres_List'] = train_df['Genres_List'].apply(ast.literal_eval)
        # Flatten list of genres
        all_genres = [genre for genres in train_df['Genres_List'] for genre in genres]
        # Count frequencies
        genre_counts = Counter(all_genres)
        # Select top N genres
        top_genres = [genre for genre, count in genre_counts.most_common(top_n)]
        print(f"Top {top_n} genres: {top_genres}")
        return top_genres
    except Exception as e:
        print(f"Error identifying top genres: {str(e)}")
        return []

def filter_genres(df, top_genres):
    """
    Filter Genres_List to include only top genres.
    Args:
        df (pd.DataFrame): DataFrame with Genres_List column.
        top_genres (list): List of genres to keep.
    Returns:
        pd.DataFrame: Filtered DataFrame with non-empty Genres_List.
    """
    df['Genres_List'] = df['Genres_List'].apply(
        lambda genres: [g for g in genres if g in top_genres]
    )
    # Remove rows with empty Genres_List
    df = df[df['Genres_List'].apply(len) > 0].copy()
    return df

def test_data_processing():
    """
    Test data processing from Task 1, checking for datasets and columns.
    """
    print("\nTesting data processing...")
    try:
        train_file = os.path.join(BASE_DIR, "data", "train_dataset.csv")
        test_file = os.path.join(BASE_DIR, "data", "test_dataset.csv")
        
        check_file_exists(train_file, "Training dataset")
        check_file_exists(test_file, "Test dataset")
        
        train_df = pd.read_csv(train_file)
        test_df = pd.read_csv(test_file)
        
        assert 'Cleaned_Summary' in train_df.columns, "Cleaned_Summary column missing in train_dataset.csv"
        assert 'Genres_List' in train_df.columns, "Genres_List column missing in train_dataset.csv"
        assert 'Cleaned_Summary' in test_df.columns, "Cleaned_Summary column missing in test_dataset.csv"
        assert 'Genres_List' in test_df.columns, "Genres_List column missing in test_dataset.csv"
        
        # Optionally check top genres
        top_genres = get_top_genres(train_file)
        assert len(top_genres) > 0, "No genres identified in training data"
        
        print("Data processing test passed: Train and test datasets exist with correct columns.")
        return True
    except Exception as e:
        print(f"Data processing test failed: {str(e)}")
        return False

def test_translation_and_audio():
    """
    Test translation and audio generation from Task 2.
    """
    print("\nTesting translation and audio generation...")
    try:
        translator = Translator()
        summary = "A brave hero fights villains to save the world."
        translated = translator.translate(summary, dest='ar').text
        assert translated, "Translation to Arabic failed"
        
        temp_dir = os.path.join(BASE_DIR, "temp")
        os.makedirs(temp_dir, exist_ok=True)
        audio_file = os.path.join(temp_dir, "test_arabic.mp3")
        
        tts = gTTS(text=translated, lang='ar')
        tts.save(audio_file)
        check_file_exists(audio_file, "Test audio file")
        
        pygame.mixer.init()
        pygame.mixer.music.load(audio_file)
        pygame.mixer.music.play()
        time.sleep(3)
        pygame.mixer.quit()
        
        if os.path.exists(audio_file):
            os.remove(audio_file)
        
        print("Translation and audio test passed: Arabic audio generated and played.")
        return True
    except Exception as e:
        print(f"Translation and audio test failed: {str(e)}")
        return False

def test_model_prediction():
    """
    Test genre prediction from Task 3, using top 20 genres.
    """
    print("\nTesting model prediction...")
    try:
        train_file = os.path.join(BASE_DIR, "data", "train_dataset.csv")
        test_file = os.path.join(BASE_DIR, "data", "test_dataset.csv")
        vectorizer_file = os.path.join(BASE_DIR, "models", "tfidf_vectorizer_top20.pkl")
        mlb_file = os.path.join(BASE_DIR, "models", "mlb_top20.pkl")
        model_file = os.path.join(BASE_DIR, "models", "logistic_regression_model_top20.pkl")
        
        # Load and filter datasets
        top_genres = get_top_genres(train_file)
        if not top_genres:
            raise ValueError("No top genres identified")
        
        train_df = pd.read_csv(train_file)
        test_df = pd.read_csv(test_file)
        train_df['Genres_List'] = train_df['Genres_List'].apply(ast.literal_eval)
        test_df['Genres_List'] = test_df['Genres_List'].apply(ast.literal_eval)
        
        train_df = filter_genres(train_df, top_genres)
        test_df = filter_genres(test_df, top_genres)
        
        if train_df.empty or test_df.empty:
            raise ValueError("Filtered datasets are empty")
        
        # Train or load TF-IDF vectorizer
        if not os.path.exists(vectorizer_file):
            vectorizer = TfidfVectorizer(max_features=5000)
            X_train = vectorizer.fit_transform(train_df['Cleaned_Summary'].fillna(''))
            with open(vectorizer_file, 'wb') as f:
                pickle.dump(vectorizer, f)
        else:
            with open(vectorizer_file, 'rb') as f:
                vectorizer = pickle.load(f)
        
        # Prepare features and labels
        X_test = vectorizer.transform(test_df['Cleaned_Summary'].fillna(''))
        mlb = MultiLabelBinarizer()
        y_train = mlb.fit_transform(train_df['Genres_List'])
        y_test = mlb.transform(test_df['Genres_List'])
        
        # Save MultiLabelBinarizer
        with open(mlb_file, 'wb') as f:
            pickle.dump(mlb, f)
        
        # Train or load model
        if not os.path.exists(model_file):
            model = OneVsRestClassifier(LogisticRegression(max_iter=1000))
            model.fit(vectorizer.transform(train_df['Cleaned_Summary'].fillna('')), y_train)
            with open(model_file, 'wb') as f:
                pickle.dump(model, f)
        else:
            with open(model_file, 'rb') as f:
                model = pickle.load(f)
        
        # Predict and evaluate
        y_pred = model.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        print(f"Model prediction test passed: Test set subset accuracy = {accuracy:.2f}")
        return True
    except Exception as e:
        print(f"Model prediction test failed: {str(e)}")
        return False

def test_gui_input():
    """
    Simulate GUI input for Task 4 (non-GUI test), using top 20 genres.
    """
    print("\nTesting GUI input simulation...")
    try:
        vectorizer_file = os.path.join(BASE_DIR, "models", "tfidf_vectorizer_top20.pkl")
        mlb_file = os.path.join(BASE_DIR, "models", "mlb_top20.pkl")
        model_file = os.path.join(BASE_DIR, "models", "logistic_regression_model_top20.pkl")
        
        check_file_exists(vectorizer_file, "TF-IDF vectorizer")
        check_file_exists(mlb_file, "MultiLabelBinarizer")
        check_file_exists(model_file, "Logistic Regression model")
        
        with open(vectorizer_file, 'rb') as f:
            vectorizer = pickle.load(f)
        with open(mlb_file, 'rb') as f:
            mlb = pickle.load(f)
        with open(model_file, 'rb') as f:
            model = pickle.load(f)
        
        summary = "A young scientist builds a time machine."
        cleaned_summary = clean_summary(summary)
        X = vectorizer.transform([cleaned_summary])
        genres = mlb.inverse_transform(model.predict(X))[0]
        assert genres, "Genre prediction failed"
        print(f"GUI input test passed: Predicted genres = {genres}")
        return True
    except Exception as e:
        print(f"GUI input test failed: {str(e)}")
        return False

if __name__ == "__main__":
    print("Running Filmception system tests (Top 20 Genres)...")
    tests = [
        ("Data Processing", test_data_processing),
        ("Translation and Audio", test_translation_and_audio),
        ("Model Prediction", test_model_prediction),
        ("GUI Input Simulation", test_gui_input)
    ]
    
    results = []
    for test_name, test_func in tests:
        result = test_func()
        results.append((test_name, result))
    
    print("\nTest Summary:")
    for test_name, passed in results:
        status = "PASSED" if passed else "FAILED"
        print(f"{test_name}: {status}")
    
    if all(passed for _, passed in results):
        print("All tests passed successfully!")
    else:
        print("Some tests failed. Check the output for details.")

Running Filmception system tests...

Testing data processing...
Data processing test passed: Train and test datasets exist with correct columns.

Testing translation and audio generation...
Translation and audio test failed: 'coroutine' object has no attribute 'text'

Testing model prediction...
Model prediction test passed: Test set accuracy = 0.08

Testing GUI input simulation...
GUI input test passed: Predicted genres = ('Science Fiction',)

Test Summary:
Data Processing: PASSED
Translation and Audio: FAILED
Model Prediction: PASSED
GUI Input Simulation: PASSED
Some tests failed. Check the output for details.


In [18]:
import os
import ast
import pickle
import pandas as pd
from extract_features import extract_tfidf_features
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import hamming_loss, accuracy_score, f1_score, classification_report
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def prepare_labels(train_file, test_file, mlb_file):
    """
    Prepare multi-label genre labels.
    Args:
        train_file (str): Path to train_dataset.csv.
        test_file (str): Path to test_dataset.csv.
        mlb_file (str): Path to save the fitted MultiLabelBinarizer.
    Returns:
        tuple: (y_train, y_test, mlb) Binary labels and binarizer.
    """
    try:
        # Load datasets
        train_df = pd.read_csv(train_file)
        test_df = pd.read_csv(test_file)
    except FileNotFoundError as e:
        logger.error(f"Error: Dataset file not found: {e}")
        raise
    except Exception as e:
        logger.error(f"Error reading dataset: {e}")
        raise

    # Validate required columns
    if 'Genres_List' not in train_df.columns or 'Genres_List' not in test_df.columns:
        logger.error("Datasets must contain 'Genres_List' column.")
        raise ValueError("Missing 'Genres_List' column.")

    # Convert string representations of lists to actual lists
    try:
        train_df['Genres_List'] = train_df['Genres_List'].apply(ast.literal_eval)
        test_df['Genres_List'] = test_df['Genres_List'].apply(ast.literal_eval)
    except (ValueError, SyntaxError) as e:
        logger.error(f"Error parsing Genres_List: {e}")
        raise

    # Initialize MultiLabelBinarizer
    mlb = MultiLabelBinarizer()

    # Fit and transform on training genres
    y_train = mlb.fit_transform(train_df['Genres_List'])

    # Transform test genres
    try:
        y_test = mlb.transform(test_df['Genres_List'])
    except ValueError as e:
        logger.error(f"Error: Test set contains unseen genres: {e}")
        raise

    # Save the binarizer
    try:
        with open(mlb_file, 'wb') as f:
            pickle.dump(mlb, f)
        logger.info(f"MultiLabelBinarizer saved to '{mlb_file}'.")
    except Exception as e:
        logger.error(f"Error saving MultiLabelBinarizer: {e}")
        raise

    return y_train, y_test, mlb

def train_model(X_train, y_train, model_file):
    """
    Train a multi-label Logistic Regression model.
    Args:
        X_train: TF-IDF features for training.
        y_train: Binary genre labels for training.
        model_file (str): Path to save the trained model.
    Returns:
        OneVsRestClassifier: Trained model.
    """
    # Initialize Logistic Regression with OneVsRestClassifier
    model = OneVsRestClassifier(LogisticRegression(max_iter=1000))

    # Train the model
    try:
        model.fit(X_train, y_train)
        logger.info("Model training completed.")
    except Exception as e:
        logger.error(f"Error training model: {e}")
        raise

    # Save the model
    try:
        with open(model_file, 'wb') as f:
            pickle.dump(model, f)
        logger.info(f"Model saved to '{model_file}'.")
    except Exception as e:
        logger.error(f"Error saving model: {e}")
        raise

    return model

def evaluate_model(model, X_test, y_test, mlb, metrics_file):
    """
    Evaluate the model on the test set and save metrics.
    Args:
        model: Trained OneVsRestClassifier.
        X_test: TF-IDF features for testing.
        y_test: Binary genre labels for testing.
        mlb: Fitted MultiLabelBinarizer.
        metrics_file (str): Path to save evaluation metrics.
    """
    # Predict on test set
    try:
        y_pred = model.predict(X_test)
    except Exception as e:
        logger.error(f"Error predicting on test set: {e}")
        raise

    # Compute metrics
    hamming = hamming_loss(y_test, y_pred)
    subset_acc = accuracy_score(y_test, y_pred)
    f1_micro = f1_score(y_test, y_pred, average='micro', zero_division=0)
    f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
    f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    # Generate classification report
    report = classification_report(y_test, y_pred, target_names=mlb.classes_, zero_division=0)

    # Log metrics
    metrics_summary = (
        f"Evaluation Metrics:\n"
        f"Hamming Loss: {hamming:.4f}\n"
        f"Subset Accuracy: {subset_acc:.4f}\n"
        f"F1 Score (Micro): {f1_micro:.4f}\n"
        f"F1 Score (Macro): {f1_macro:.4f}\n"
        f"F1 Score (Weighted): {f1_weighted:.4f}\n\n"
        f"Classification Report:\n{report}"
    )
    logger.info(metrics_summary)

    # Save metrics to file
    try:
        with open(metrics_file, 'w') as f:
            f.write(metrics_summary)
        logger.info(f"Metrics saved to '{metrics_file}'.")
    except Exception as e:
        logger.error(f"Error saving metrics: {e}")
        raise

    return {
        'hamming_loss': hamming,
        'subset_accuracy': subset_acc,
        'f1_micro': f1_micro,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted
    }

if __name__ == "__main__":
    train_file = "data/train_dataset.csv"
    test_file = "data/test_dataset.csv"
    mlb_file = "models/mlb.pkl"
    model_file = "models/logistic_regression_model.pkl"
    metrics_file = "models/evaluation_metrics.txt"

    os.makedirs("models", exist_ok=True)

    # Load TF-IDF features
    try:
        X_train, X_test, vectorizer = extract_tfidf_features(train_file, test_file, "models/tfidf_vectorizer.pkl")
        logger.info(f"TF-IDF features loaded: X_train shape = {X_train.shape}, X_test shape = {X_test.shape}")
    except Exception as e:
        logger.error(f"Error extracting TF-IDF features: {e}")
        raise

    # Prepare labels
    try:
        y_train, y_test, mlb = prepare_labels(train_file, test_file, mlb_file)
        logger.info(f"Labels prepared: y_train shape = {y_train.shape}, y_test shape = {y_test.shape}")
    except Exception as e:
        logger.error(f"Error preparing labels: {e}")
        raise

    # Train model
    try:
        model = train_model(X_train, y_train, model_file)
        logger.info(f"Model trained and saved to '{model_file}'.")
    except Exception as e:
        logger.error(f"Error training model: {e}")
        raise

    # Evaluate model
    try:
        metrics = evaluate_model(model, X_test, y_test, mlb, metrics_file)
        logger.info("Model evaluation completed.")
    except Exception as e:
        logger.error(f"Error evaluating model: {e}")
        raise

2025-05-06 13:55:40,638 - INFO - TF-IDF features loaded: X_train shape = (33763, 5000), X_test shape = (8441, 5000)
2025-05-06 13:55:42,679 - INFO - MultiLabelBinarizer saved to 'models/mlb.pkl'.
2025-05-06 13:55:42,702 - INFO - Labels prepared: y_train shape = (33763, 221), y_test shape = (8441, 221)
2025-05-06 13:56:45,422 - INFO - Model training completed.
2025-05-06 13:56:45,459 - INFO - Model saved to 'models/logistic_regression_model.pkl'.
2025-05-06 13:56:45,467 - INFO - Model trained and saved to 'models/logistic_regression_model.pkl'.
2025-05-06 13:56:47,430 - INFO - Evaluation Metrics:
Hamming Loss: 0.0118
Subset Accuracy: 0.0846
F1 Score (Micro): 0.4252
F1 Score (Macro): 0.0328
F1 Score (Weighted): 0.3720

Classification Report:
                    precision    recall  f1-score   support

            Action       0.63      0.15      0.24       954
         Adventure       0.63      0.17      0.27       692
         Animation       0.79      0.11      0.19       806
         